# 04b · Spark MLlib model: Random Forest

**Task.** Binary classification: predict whether a driver files a claim (`OUTCOME = 1`).

**Why this model.** Random forest was one of the Phase 2 models, so the Spark version can be compared directly with the scikit-learn one on the same test drivers. It averages many decision trees, each grown on a bootstrap sample with a random subset of features at every split. That lets it capture non-linear effects and interactions (for example, driving experience together with vehicle ownership) that logistic regression cannot, while averaging keeps the variance of a single deep tree under control. Its feature importances show which inputs the trees rely on.

**Inputs.** `workspace.car_insurance.claims_gold_features` and `workspace.car_insurance.claims_model_split`.

**Outputs.** Fitted pipeline saved to `/Volumes/workspace/car_insurance/models/random_forest`; test-set metrics and the comparison with Phase 2, shown below.

**How to run.** Run after `03b_delta_tables`. Attach Serverless compute (environment version 4 or later) and choose **Run all**. The first code cell loads `00_setup`, which must be in the same folder.

**Assumptions.** Notebooks 01 to 03b have been run. Every Gold column other than `ID` and `OUTCOME` is a model input.

In [0]:
%run ./00_setup

# 00 · Shared setup: storage, table names and Delta helpers

**Purpose.** One place for everything the other notebooks share: where the data lives, what each Delta table is called, how tables are written and verified, and how models are split and scored. Keeping it here means all team members write to the same tables in the same way.

**How it is used.** This notebook is not run on its own. Every other notebook starts with a cell containing only `%run ./00_setup`, which runs these cells and makes the names and functions below available.

**Inputs.** None.

**Outputs.** No tables. It creates the schema `workspace.car_insurance` and the volumes `raw` (source files) and `models` (saved Spark ML models) if they do not exist yet.

**Assumptions**

- The catalog is named `workspace` (the Databricks Free Edition default). If yours differs, change `CATALOG` below; nothing else needs editing.
- Serverless compute.

**What it provides**

| Name | Use |
|---|---|
| `BRONZE_TABLE`, `SILVER_TABLE`, `GOLD_FEATURES_TABLE`, `GOLD_SUMMARY_TABLE`, `SPLIT_TABLE` | Full table names. Always use these instead of typing a table name |
| `CLAIMS_CSV`, `SPLIT_CSV`, `MODEL_DIR` | Paths inside the volumes |
| `require_source_files(...)` | Stops with a clear message if an uploaded file is missing |
| `write_delta(df, table_name)` | Saves a DataFrame as a Delta table |
| `validate_tables({...})` | Reads tables back and checks format, rows, columns and key |
| `load_model_splits()` | Train, validation and test DataFrames with the Phase 2 split |
| `evaluate_classifier(predictions)` | Accuracy, precision, recall, F1, ROC-AUC and confusion counts |

Setup loaded: tables in workspace.car_insurance, files in /Volumes/workspace/car_insurance


## 1. Data and split

`load_model_splits()` returns the same train (6,400), validation (1,600) and test (2,000) drivers as Phase 2. The model is fitted on train, its settings are chosen on validation, and test is scored once at the end.

Text columns (and `POSTAL_CODE`, an area label) are categorical inputs; numeric columns are numeric inputs. The only change to the data here is type alignment: nothing is fitted in this cell.

In [0]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler

NUMERIC_TYPES = ("tinyint", "smallint", "int", "bigint", "float", "double", "decimal", "boolean")
ALWAYS_CATEGORICAL = {"POSTAL_CODE"}   # an area label, not a quantity

gold_types = dict(spark.table(GOLD_FEATURES_TABLE).drop("split").dtypes)
candidates = [c for c in gold_types if c not in (ID_COLUMN, LABEL_COLUMN) and not c.startswith("_")]
CATEGORICAL_FEATURES = [c for c in candidates if gold_types[c] == "string" or c in ALWAYS_CATEGORICAL]
NUMERIC_FEATURES = [c for c in candidates
                    if c not in CATEGORICAL_FEATURES and gold_types[c].startswith(NUMERIC_TYPES)]
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}):", CATEGORICAL_FEATURES)
print(f"Numeric features ({len(NUMERIC_FEATURES)}):", NUMERIC_FEATURES)


def prepare(df):
    """Type alignment only: label and numbers as double, categories as text."""
    return df.select(
        ID_COLUMN,
        F.col(LABEL_COLUMN).cast("double").alias("label"),
        *[F.col(c).cast("string").alias(c) for c in CATEGORICAL_FEATURES],
        *[F.col(c).cast("double").alias(c) for c in NUMERIC_FEATURES],
    )


train_df, validation_df, test_df = (prepare(split) for split in load_model_splits())

split_stats = [(name, split.agg(F.count("*").alias("drivers"), F.sum("label").alias("claims")).first())
               for name, split in zip(SPLIT_NAMES, (train_df, validation_df, test_df))]
display(spark.createDataFrame(
    [(name, int(r["drivers"]), int(r["claims"]), round(r["claims"] / r["drivers"], 4)) for name, r in split_stats],
    "split string, drivers long, claims long, claim_rate double"))
N_TRAIN = int(split_stats[0][1]["drivers"])

Categorical features (10): ['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'VEHICLE_YEAR', 'POSTAL_CODE', 'VEHICLE_TYPE', 'CAR_AGE']
Numeric features (10): ['CREDIT_SCORE', 'VEHICLE_OWNERSHIP', 'MARRIED', 'CHILDREN', 'ANNUAL_MILEAGE', 'SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS', 'INSURANCE_RISK', 'FAMILY']


split,drivers,claims,claim_rate
train,6400,1982,0.3097
validation,1600,495,0.3094
test,2000,656,0.328


## 2. Pipeline and training

| Stage | What it does | Fitted on train |
|---|---|---|
| `StringIndexer` | Maps each category to an index. The index carries category metadata, so the trees split on groups of categories rather than treating the index as a number | The list of labels |
| `Imputer` (median) | Fills missing `CREDIT_SCORE` and `ANNUAL_MILEAGE` | The medians |
| `VectorAssembler` | Combines all inputs into one feature vector | Nothing |
| `RandomForestClassifier` | The forest | The trees |

Trees need no one-hot encoding or scaling, because a split depends only on the order of values. Every fitted step learns from the training drivers only, so no validation or test information reaches the model.

**Settings from Phase 2** (5-fold CV in scikit-learn): 300 trees, half of the features considered at each split, at least 1 driver per leaf, maximum depth 8. They are kept fixed here, and the depth is searched again on the validation split (next section).

In [0]:
SEED = 123   # same seed as Phase 2

FIXED_PARAMS = {
    "numTrees": 300,                  # Phase 2: n_estimators = 300
    "featureSubsetStrategy": "0.5",   # Phase 2: max_features = 0.5
    "minInstancesPerNode": 1,         # Phase 2: min_samples_leaf = 1
    "seed": SEED,
}


def build_pipeline(max_depth):
    indexed = [f"{c}_index" for c in CATEGORICAL_FEATURES]
    imputed = [f"{c}_imputed" for c in NUMERIC_FEATURES]
    return Pipeline(stages=[
        # handleInvalid="error": a category never seen in training stops the run instead of being guessed.
        StringIndexer(inputCols=CATEGORICAL_FEATURES, outputCols=indexed,
                      stringOrderType="alphabetDesc", handleInvalid="error"),
        Imputer(inputCols=NUMERIC_FEATURES, outputCols=imputed, strategy="median"),
        VectorAssembler(inputCols=indexed + imputed, outputCol="features"),
        RandomForestClassifier(featuresCol="features", labelCol="label", maxDepth=max_depth, **FIXED_PARAMS),
    ])


## 3. Choosing the depth on the validation split

Maximum depth controls how complex each tree can be: deeper trees fit the training drivers more closely but can overfit. Depths 4, 6, 8 (the Phase 2 choice) and 10 are each fitted on train and scored on validation. The depth with the highest validation ROC-AUC is kept (Phase 2 also tuned by ROC-AUC); a tie goes to the shallower, simpler forest. One validation split is used instead of Phase 2's 5-fold cross-validation, to keep the number of Spark fits small on shared serverless compute.

In [0]:
import time

DEPTHS = [4, 6, 8, 10]
tuning_rows, best = [], None
for depth in DEPTHS:
    start = time.perf_counter()
    candidate = build_pipeline(depth).fit(train_df)
    fit_seconds = time.perf_counter() - start
    train_m = evaluate_classifier(candidate.transform(train_df), "label")
    val_m = evaluate_classifier(candidate.transform(validation_df), "label")
    tuning_rows.append((depth, round(fit_seconds, 1), round(train_m["accuracy"], 4), round(val_m["accuracy"], 4),
                        round(train_m["roc_auc"], 4), round(val_m["roc_auc"], 4)))
    if best is None or val_m["roc_auc"] > best["val_roc_auc"]:
        best = {"depth": depth, "model": candidate, "fit_seconds": fit_seconds, "val_roc_auc": val_m["roc_auc"]}
    del candidate   # serverless caps the total size of models held in memory

display(spark.createDataFrame(
    tuning_rows, "max_depth int, fit_seconds double, train_accuracy double, validation_accuracy double, "
                 "train_roc_auc double, validation_roc_auc double"))

model = best["model"]
print(f"Selected maxDepth = {best['depth']} (validation ROC-AUC {best['val_roc_auc']:.4f}, "
      f"fit {best['fit_seconds']:.1f} s on {N_TRAIN:,} drivers)")

max_depth,fit_seconds,train_accuracy,validation_accuracy,train_roc_auc,validation_roc_auc
4,8.0,0.8192,0.8225,0.901,0.8991
6,11.4,0.8581,0.85,0.9245,0.9166
8,18.9,0.8748,0.8562,0.9476,0.9172
10,28.9,0.912,0.8512,0.972,0.9153


Selected maxDepth = 8 (validation ROC-AUC 0.9172, fit 18.9 s on 6,400 drivers)


**Reading the table.** The gap between train and validation scores is the sign of overfitting: it grows as the trees get deeper. The selected depth is the one where validation ROC-AUC is highest; deeper trees beyond it fit the training drivers better without improving on unseen ones.

## 4. Test-set evaluation

The selected forest is scored once on the 2,000 test drivers. Precision, recall and F1 are for the claim class, as in Phase 2. A driver is predicted to claim when the forest's probability is at least 0.5.

In [0]:
test_metrics = evaluate_classifier(model.transform(test_df), "label")

display(spark.createDataFrame(
    [(m, round(test_metrics[m], 4)) for m in ("accuracy", "precision", "recall", "f1", "roc_auc")],
    "metric string, test_value double"))

display(spark.createDataFrame(
    [("Actual: no claim", test_metrics["tn"], test_metrics["fp"]),
     ("Actual: claim", test_metrics["fn"], test_metrics["tp"])],
    "actual string, predicted_no_claim long, predicted_claim long"))

metric,test_value
accuracy,0.8545
precision,0.7821
recall,0.7713
f1,0.7767
roc_auc,0.9185


actual,predicted_no_claim,predicted_claim
Actual: no claim,1203,141
Actual: claim,150,506


## 5. Comparison with Phase 2

The Phase 2 numbers are the tuned scikit-learn random forest on the same 2,000 test drivers (Phase 2 report, Table VII). The first row is the trivial baseline that always predicts "no claim".

In [0]:
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc"]
PHASE2_RF = {"accuracy": 0.8610, "precision": 0.7980, "recall": 0.7450, "f1": 0.7710, "roc_auc": 0.9250}

test_total = sum(test_metrics[k] for k in ("tp", "fp", "tn", "fn"))
baseline = {"accuracy": (test_metrics["tn"] + test_metrics["fp"]) / test_total,
            "precision": 0.0, "recall": 0.0, "f1": 0.0, "roc_auc": 0.5}

display(spark.createDataFrame(
    [(name, *[round(float(values[m]), 4) for m in METRICS])
     for name, values in [("Always 'no claim' (this test set)", baseline),
                          ("Phase 2 scikit-learn random forest (tuned)", PHASE2_RF),
                          ("Phase 3 Spark MLlib random forest", test_metrics)]],
    "model string, accuracy double, precision double, recall double, f1 double, roc_auc double"))

display(spark.createDataFrame(
    [(m, round(test_metrics[m] - PHASE2_RF[m], 4)) for m in METRICS],
    "metric string, spark_minus_phase2 double"))

model,accuracy,precision,recall,f1,roc_auc
Always 'no claim' (this test set),0.672,0.0,0.0,0.0,0.5
Phase 2 scikit-learn random forest (tuned),0.861,0.798,0.745,0.771,0.925
Phase 3 Spark MLlib random forest,0.8545,0.7821,0.7713,0.7767,0.9185


metric,spark_minus_phase2
accuracy,-0.0065
precision,-0.0159
recall,0.0263
f1,0.0057
roc_auc,-0.0065


### Why the numbers can differ

Both versions are tested on the same 2,000 drivers, so differences come from how the model is built. One percentage point of accuracy is 20 test drivers.

| Aspect | Phase 2 (scikit-learn) | Phase 3 (Spark MLlib) |
|---|---|---|
| Training drivers | 8,000 (train + validation) | 6,400 (train) |
| Tuning | 5-fold cross-validation grid search | One validation split, depth only |
| Features | 29 one-hot encoded columns | Gold columns; categories indexed, not one-hot |
| Missing values | Linear-regression imputation | Training median |
| Split candidates | Every distinct value of a feature | At most 32 bins per feature (`maxBins`) |
| Bootstrap and feature sampling | `random_state=123` | `seed=123` (a different random generator) |

The biggest expected effects are the 1,600 fewer training drivers and Spark's binning of continuous features such as `CREDIT_SCORE` and `ANNUAL_MILEAGE` into 32 candidate split points.

## 6. Feature importance

Impurity-based importance from the fitted forest: how much each input reduces node impurity across all trees, normalized to sum to 1. A categorical input counts once, because it is indexed rather than one-hot encoded. In Phase 2, driving experience, vehicle ownership, vehicle year and postal code were the leading features.

In [0]:
forest = model.stages[-1]
importances = [float(v) for v in forest.featureImportances.toArray()]
feature_names = CATEGORICAL_FEATURES + NUMERIC_FEATURES   # same order as the VectorAssembler inputs
assert len(importances) == len(feature_names), "Feature names do not line up with the importances"

display(spark.createDataFrame(
    sorted(zip(feature_names, [round(v, 4) for v in importances]), key=lambda row: -row[1]),
    "feature string, importance double"))

feature,importance
DRIVING_EXPERIENCE,0.289
VEHICLE_OWNERSHIP,0.1419
INSURANCE_RISK,0.1227
VEHICLE_YEAR,0.0894
POSTAL_CODE,0.075
AGE,0.0695
GENDER,0.0341
CREDIT_SCORE,0.0322
ANNUAL_MILEAGE,0.0309
INCOME,0.0264


## 7. Save and reload the model

The fitted pipeline (indexer, imputer, assembler and forest) is saved to the `models` volume, loaded back, and checked to give exactly the same test predictions.

In [0]:
MODEL_PATH = f"{MODEL_DIR}/random_forest"
model.write().overwrite().save(MODEL_PATH)
print("Saved to", MODEL_PATH)

reloaded_metrics = evaluate_classifier(PipelineModel.load(MODEL_PATH).transform(test_df), "label")
for name in ("tp", "fp", "tn", "fn"):
    assert reloaded_metrics[name] == test_metrics[name], "The reloaded model predicts differently"
print(f"Reloaded model reproduces the test results: accuracy {reloaded_metrics['accuracy']:.4f}, "
      f"ROC-AUC {reloaded_metrics['roc_auc']:.4f}")

Saved to /Volumes/workspace/car_insurance/models/random_forest
Reloaded model reproduces the test results: accuracy 0.8545, ROC-AUC 0.9185
